# Notebook 02: Data Cleaning & ETL (Pure Parquet)
## Retail Demand Forecasting & Inventory Optimization

**Objective:** Convert M5 wide-format sales to long format and save as Parquet files for fast re-use.

**No database required.** All outputs go to `Data/processed/`.

**Outputs:**
- `daily_aggregated.parquet` - Date × Category × Store aggregations (used by EDA & Dashboard)
- `sales_enriched_sample.parquet` - Top 200 SKUs fully enriched (used by Forecasting & Inventory)
- `calendar_clean.parquet` - Clean calendar with all event/SNAP features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import time, warnings
warnings.filterwarnings('ignore')

# ── PATHS ─────────────────────────────────────────────────────────
DATA_RAW  = Path('../Data')            # M5 CSV files
DATA_PROC = Path('../Data/processed')  # Parquet outputs
DATA_PROC.mkdir(parents=True, exist_ok=True)

# Verify files
for f in ['sales_train_validation.csv', 'calendar.csv', 'sell_prices.csv']:
    path = DATA_RAW / f
    status = f'OK   {path.stat().st_size/1e6:.0f} MB' if path.exists() else 'MISSING'
    print(f'  {status}  {f}')

print('\nEnvironment ready')

## Step 1: Load Raw Files

In [ ]:
print('Loading raw files ...')

sales    = pd.read_csv(DATA_RAW / 'sales_train_validation.csv')
calendar = pd.read_csv(DATA_RAW / 'calendar.csv', parse_dates=['date'])
prices   = pd.read_csv(DATA_RAW / 'sell_prices.csv')

id_cols  = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales.columns if c.startswith('d_')]

print(f'Sales    : {sales.shape[0]:,} SKU-Store rows x {sales.shape[1]} cols')
print(f'Calendar : {calendar.shape[0]:,} days | {calendar.date.min().date()} to {calendar.date.max().date()}')
print(f'Prices   : {prices.shape[0]:,} rows')
print(f'Day cols : {len(day_cols)} ({day_cols[0]} to {day_cols[-1]})')

## Step 2: Save Clean Calendar

In [ ]:
# Keep only needed calendar columns
cal = calendar[[
    'date', 'd', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
    'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
    'snap_CA', 'snap_TX', 'snap_WI'
]].copy()

cal.to_parquet(DATA_PROC / 'calendar_clean.parquet', index=False)
print(f'Saved calendar_clean.parquet: {len(cal):,} rows')
display(cal.head(3))

## Step 3: Identify Top 200 SKUs

In [ ]:
# Identify top 200 SKUs by total lifetime sales (for sample parquet)
total_sales = sales[day_cols].sum(axis=1)
top_200_ids = set(sales.loc[total_sales.nlargest(200).index, 'id'])

print(f'Top 200 SKUs identified')
print(f'Min sales in top-200: {total_sales.nlargest(200).min():,}')

# Build a date lookup: d_1 -> date
date_lookup = dict(zip(calendar['d'], calendar['date']))
wk_lookup   = dict(zip(calendar['d'], calendar['wm_yr_wk']))

print('Lookups built')

## Step 4: Melt Wide → Long (Chunked)

> Processes 300 SKUs at a time to stay within RAM limits.
> Builds two outputs simultaneously: the daily aggregation and the top-200 sample.
>
> Estimated runtime: **5–15 minutes** depending on your machine.

In [ ]:
start = time.time()
BATCH = 300  # SKUs per chunk

# Accumulators
sample_chunks = []   # top-200 SKUs, fully enriched
agg_chunks    = []   # daily aggregations for all SKUs

for i in tqdm(range(0, len(sales), BATCH), desc='Melting sales wide->long'):
    chunk = sales.iloc[i : i + BATCH]

    # ── Melt ────────────────────────────────────────────────────────
    melted = chunk.melt(
        id_vars=id_cols,
        value_vars=day_cols,
        var_name='d',
        value_name='units_sold'
    )
    melted['units_sold'] = melted['units_sold'].fillna(0).astype(np.int32)

    # ── Join calendar features ───────────────────────────────────────
    cal_cols_needed = ['d', 'date', 'wm_yr_wk', 'year', 'month', 'weekday', 'wday',
                       'event_name_1', 'event_type_1',
                       'snap_CA', 'snap_TX', 'snap_WI']
    melted = melted.merge(cal[cal_cols_needed], on='d', how='left')

    # ── Join prices ──────────────────────────────────────────────────
    melted = melted.merge(
        prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
        on=['store_id', 'item_id', 'wm_yr_wk'],
        how='left'
    )

    # ── 1. Daily aggregation (all SKUs) ──────────────────────────────
    melted['revenue'] = melted['units_sold'] * melted['sell_price'].fillna(0)
    agg = melted.groupby(
        ['date', 'cat_id', 'dept_id', 'store_id', 'state_id',
         'year', 'month', 'weekday', 'wday', 'event_name_1',
         'snap_CA', 'snap_TX', 'snap_WI'],
        dropna=False
    ).agg(total_units=('units_sold','sum'), total_revenue=('revenue','sum')).reset_index()
    agg_chunks.append(agg)

    # ── 2. Top-200 sample ────────────────────────────────────────────
    mask = melted['id'].isin(top_200_ids)
    if mask.any():
        sample = melted.loc[mask, [
            'date', 'id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id',
            'year', 'month', 'weekday', 'wday',
            'event_name_1', 'snap_CA', 'snap_TX', 'snap_WI',
            'units_sold', 'sell_price'
        ]].copy()
        sample = sample.rename(columns={'id': 'product_id', 'units_sold': 'units_sold'})
        sample_chunks.append(sample)

elapsed = time.time() - start
print(f'\nMelt complete in {elapsed/60:.1f} minutes')
print(f'  Aggregation chunks : {len(agg_chunks)}')
print(f'  Sample chunks      : {len(sample_chunks)}')

## Step 5: Consolidate & Save

In [ ]:
# ── Daily Aggregated ────────────────────────────────────────────────
print('Consolidating daily aggregations ...')
daily_agg = pd.concat(agg_chunks, ignore_index=True)

# Re-aggregate (chunks overlap on date+category+store combinations)
daily_agg = daily_agg.groupby(
    ['date', 'cat_id', 'dept_id', 'store_id', 'state_id',
     'year', 'month', 'weekday', 'wday', 'event_name_1',
     'snap_CA', 'snap_TX', 'snap_WI'],
    dropna=False
).agg(total_units=('total_units','sum'), total_revenue=('total_revenue','sum')).reset_index()

daily_agg.to_parquet(DATA_PROC / 'daily_aggregated.parquet', index=False)
print(f'Saved daily_aggregated.parquet: {len(daily_agg):,} rows')

# Free memory
del agg_chunks

# ── Enriched Sample ─────────────────────────────────────────────────
print('\nConsolidating enriched sample ...')
df_sample = pd.concat(sample_chunks, ignore_index=True)
df_sample.to_parquet(DATA_PROC / 'sales_enriched_sample.parquet', index=False)
print(f'Saved sales_enriched_sample.parquet: {len(df_sample):,} rows')
print(f'  SKUs in sample: {df_sample["product_id"].nunique()}')

del sample_chunks
print('\nAll parquet files saved to Data/processed/')

## Step 6: Verification

In [ ]:
print('=' * 55)
print('  PARQUET OUTPUT VERIFICATION')
print('=' * 55)

for fname in ['calendar_clean.parquet', 'daily_aggregated.parquet', 'sales_enriched_sample.parquet']:
    path = DATA_PROC / fname
    if path.exists():
        df = pd.read_parquet(path)
        size_mb = path.stat().st_size / 1e6
        print(f'  OK  {fname}')
        print(f'      Rows: {len(df):,} | Size: {size_mb:.1f} MB | Cols: {list(df.columns[:5])}...')
    else:
        print(f'  MISSING  {fname}')

# Quick sanity check
print('\nSanity checks on daily_aggregated:')
agg = pd.read_parquet(DATA_PROC / 'daily_aggregated.parquet')
print(f'  Date range : {agg.date.min().date()} to {agg.date.max().date()}')
print(f'  Categories : {sorted(agg.cat_id.unique())}')
print(f'  Stores     : {sorted(agg.store_id.unique())}')
print(f'  Total units: {agg.total_units.sum():,}')
print(f'  Total rev  : ${agg.total_revenue.sum():,.0f}')

print('\nETL Complete. Proceed to Notebook 03 (EDA).')